# PyTorch Basics — MNIST classification

**Companion notebook** to the lecture slides.

> Run every cell, top to bottom. Each section ends with a small **sanity
> check** so you can confirm shapes, dtypes, and devices match what the
> slides showed.

---

## Contents

1. PyTorch setup & GPU check
2. Tensors
3. Autograd
4. Building a model with `nn.Module`
5. Loading MNIST with `Dataset` & `DataLoader`
6. Loss & optimizer
7. The training loop (with a sanity check first)
8. Evaluating and visualizing predictions
9. Debugging cheatsheet

The MNIST dataset will be downloaded automatically (~12 MB) the first time
you run section 5.


## 1. PyTorch setup & GPU check

PyTorch is pre-installed on Colab. Just import it and check the version /
GPU. If `torch.cuda.is_available()` returns `False`, switch to a GPU runtime
or continue on CPU.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device          :", torch.cuda.get_device_name(0))


: 

**The device pattern.** Pick the device once at the top, then call
`.to(device)` on every tensor and the model.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## 2. Tensors

If you've used NumPy, you already know 90% of this. The remaining 10% is
about `dtype`, `device`, and the image-batch convention `(B, C, H, W)`.

### 2.1 Creating tensors

In [ ]:
# From Python data
a = torch.tensor([[1, 2], [3, 4]])
print(a, a.dtype)   # int64 by default

# Filled tensors
zeros = torch.zeros(3, 4)            # (3, 4) float32
ones  = torch.ones(2, 3)
rnd   = torch.randn(2, 3)            # standard normal

print("zeros:", zeros.shape, zeros.dtype)
print("rnd  :", rnd)


In [ ]:
# From / to NumPy (shares memory!)
arr = np.arange(6).reshape(2, 3)
t = torch.from_numpy(arr).float()
print("tensor :", t)

t[0, 0] = 99      # mutating the tensor...
print("numpy  :", arr)   # ...also changed the numpy array


### 2.2 Shape, dtype, device — and (B, C, H, W)

For images, PyTorch uses `(batch, channels, height, width)`. Channel comes
**before** the spatial axes — unlike NumPy/PIL which usually put it last.

For MNIST specifically: each image is `(1, 28, 28)` — one grayscale channel,
28 by 28 pixels.

In [ ]:
x = torch.randn(8, 1, 28, 28)   # 8 grayscale 28x28 images

print("shape :", x.shape)
print("dtype :", x.dtype)
print("device:", x.device)

# Reshape / view operations
y = x.view(8, -1)         # flatten per sample: (8, 784)
u = x.unsqueeze(0)        # add a leading dim:  (1, 8, 1, 28, 28)
v = x[0].squeeze()        # drop size-1 dims

print("flat  :", y.shape)
print("u     :", u.shape)


### 2.3 Operations, broadcasting, GPU

In [ ]:
a = torch.randn(4, 3)
b = torch.randn(3, 5)

print("a + 1     :", (a + 1).shape)
print("a.mean()  :", a.mean().item())
print("a.sum(1)  :", a.sum(dim=1).shape)
print("a @ b     :", (a @ b).shape)        # matrix multiply


In [ ]:
# Broadcasting: per-channel normalization (this is what MNIST's
# transforms.Normalize does under the hood)
img  = torch.randn(1, 28, 28)
mean = torch.tensor([0.1307]).view(1, 1, 1)
std  = torch.tensor([0.3081]).view(1, 1, 1)

img_norm = (img - mean) / std         # broadcasts over H, W
print("normalized:", img_norm.shape)


In [ ]:
# Move to GPU (if available)
img = img.to(device)
print("img is on:", img.device)

# Operations preserve device
img2 = img * 2.0
print("img2 is on:", img2.device)


**Sanity check:** the prints above should show `(B, C, H, W)` shapes
where appropriate, and the same device for tensors involved in the same op.

## 3. Autograd

PyTorch records operations on tensors with `requires_grad=True` into a
computation graph, then computes gradients via `.backward()`. You almost
never set `requires_grad` manually — model parameters set it automatically.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3 + 2 * x          # y = x^3 + 2x
print("y =", y.item())

y.backward()                # populate x.grad
print("dy/dx =", x.grad.item())   # 3*x^2 + 2 = 14.0


At evaluation time, disable gradient tracking with `torch.no_grad()` —
it's faster and uses less memory.

In [ ]:
with torch.no_grad():
    a = torch.randn(2, 3, requires_grad=True)
    b = a ** 2
    print("requires_grad inside no_grad:", b.requires_grad)


## 4. Building a model with `nn.Module`

Every model in PyTorch is an `nn.Module`. You only need two methods:

- `__init__` — register layers as attributes
- `forward` — define how input flows through them

`nn.Module` then handles parameter tracking, device transfer, save/load,
gradient hooks, and the train/eval mode switch for you.

### 4.1 A simple MLP for MNIST

A 3-layer multi-layer perceptron: flatten the 28×28 image to 784 features,
then two hidden layers, then a 10-dimensional output (one logit per digit).

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)        # (B, 784)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)               # logits, shape (B, 10)


model = MLP()
print(model)
print("\nparams:", sum(p.numel() for p in model.parameters()))


**Important:** call `model(x)`, **not** `model.forward(x)`. The
`__call__` wrapper runs the hooks PyTorch needs (e.g., for autograd).

In [ ]:
# Forward pass shape check
x = torch.randn(2, 1, 28, 28)
out = model(x)
print("input :", x.shape)
print("output:", out.shape)   # (2, 10)


Move the model to the device once. From now on, every input we feed it
must also be on the same device.

In [ ]:
model = MLP().to(device)
print("model is on:", next(model.parameters()).device)


## 5. Loading MNIST

`torchvision` provides MNIST as a standard `Dataset`. The first run
downloads ~12 MB of data into `./data/`.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),                            # PIL -> (1, 28, 28) in [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)),       # MNIST mean / std
])

train_ds = datasets.MNIST(root="./data", train=True,
                          download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False,
                          download=True, transform=transform)

print("train size:", len(train_ds))
print("test size :", len(test_ds))


### 5.1 Inspect a single sample

`Dataset` is just a container that returns `(sample, label)` for a given
index. Three methods power it: `__init__`, `__len__`, `__getitem__`.

In [ ]:
img, label = train_ds[0]
print("img   :", img.shape, img.dtype)   # torch.Size([1, 28, 28]) torch.float32
print("label :", label)                  # 5

# Visualize the first 8 samples
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for i in range(8):
    img_i, lbl_i = train_ds[i]
    # Reverse the normalization for display
    axes[i].imshow(img_i.squeeze().numpy() * 0.3081 + 0.1307, cmap="gray")
    axes[i].set_title(str(lbl_i))
    axes[i].axis("off")
plt.tight_layout(); plt.show()


### 5.2 Wrap with `DataLoader`

`DataLoader` adds batching, shuffling, and parallel loading.

In [ ]:
train_loader = DataLoader(train_ds, batch_size=64,  shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)

# Inspect one batch
images, labels = next(iter(train_loader))
print("batch images:", images.shape, images.dtype)   # (64, 1, 28, 28) float32
print("batch labels:", labels.shape, labels.dtype)   # (64,) int64 (long)
print("first labels:", labels[:8].tolist())


## 6. Loss & optimizer

For 10-class classification, **`CrossEntropyLoss`** takes raw logits — do
**not** apply softmax yourself; it's done internally in a numerically stable
way.

- `logits` shape: `(B, num_classes)`
- `labels` shape: `(B,)`, dtype `long` (integer class indices)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("loss + optimizer ready")


## 7. The training loop

The five-step train step:

1. `optimizer.zero_grad()` — clear stale gradients
2. forward — `logits = model(x)`
3. compute the loss
4. `loss.backward()` — backward pass
5. `optimizer.step()` — update weights

Wrapped in an epoch loop and a `model.train()` / `model.eval()` switch.

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()              # 1
        logits = model(images)             # 2
        loss   = criterion(logits, labels) # 3
        loss.backward()                    # 4
        optimizer.step()                   # 5

        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        preds  = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total


### 7.1 Sanity check: overfit a single batch

Before training on the full dataset, prove the pipeline works by overfitting
one batch. If loss can't drop close to zero, the model or pipeline is broken
— go fix that before wasting epochs.

In [ ]:
sanity_model = MLP().to(device)
sanity_opt   = torch.optim.Adam(sanity_model.parameters(), lr=1e-3)

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

sanity_model.train()
for step in range(80):
    sanity_opt.zero_grad()
    loss = criterion(sanity_model(images), labels)
    loss.backward()
    sanity_opt.step()
    if step % 20 == 0 or step == 79:
        print(f"step {step:3d}  loss {loss.item():.4f}")


You should see the loss drop sharply (well below 0.1, often near 0).
If it doesn't, double-check shapes, dtypes, and device.

### 7.2 Now train for real

In [ ]:
NUM_EPOCHS = 3   # raise this if you want a higher score

# Reinitialize so we start from scratch
model     = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_history, test_acc_history = [], []
best_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_acc   = evaluate(model, test_loader, device)
    train_history.append(train_loss)
    test_acc_history.append(test_acc)

    flag = ""
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), "best.pt")
        flag = "  ← saved"

    print(f"epoch {epoch}/{NUM_EPOCHS}  "
          f"train loss {train_loss:.4f}  test acc {test_acc:.4f}{flag}")


In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].plot(train_history, marker="o"); axes[0].set_title("Train loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
axes[1].plot(test_acc_history, marker="o", color="C2"); axes[1].set_title("Test accuracy")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].set_ylim(0.9, 1.0)
plt.tight_layout(); plt.show()


## 8. Evaluating & visualizing predictions

Reload the best checkpoint and look at a few predictions, including any
misclassifications.

In [ ]:
# Reload the best checkpoint to demonstrate save/load
model.load_state_dict(torch.load("best.pt", map_location=device))
model.eval()

with torch.no_grad():
    images, labels = next(iter(test_loader))
    images_dev = images.to(device)
    logits = model(images_dev)
    preds  = logits.argmax(dim=1).cpu()

# Show 12 random samples with predicted vs true labels
idx = np.random.choice(len(images), 12, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for k, i in enumerate(idx):
    ax = axes[k // 6, k % 6]
    ax.imshow(images[i].squeeze().numpy() * 0.3081 + 0.1307, cmap="gray")
    correct = preds[i].item() == labels[i].item()
    color = "green" if correct else "red"
    ax.set_title(f"pred {preds[i].item()} / true {labels[i].item()}", color=color, fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# Find misclassified examples specifically
wrong = (preds != labels).nonzero(as_tuple=True)[0]
print(f"{len(wrong)} misclassified out of {len(labels)} in this batch")

if len(wrong) > 0:
    fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(10, 1.7))
    if min(8, len(wrong)) == 1:
        axes = [axes]
    for k, i in enumerate(wrong[:8]):
        axes[k].imshow(images[i].squeeze().numpy() * 0.3081 + 0.1307, cmap="gray")
        axes[k].set_title(f"{preds[i].item()} ≠ {labels[i].item()}", color="red", fontsize=10)
        axes[k].axis("off")
    plt.tight_layout(); plt.show()


## 9. Debugging cheatsheet

A few one-liners that will save you hours during the assignment.

In [ ]:
# Inspect a tensor — print early, print often
x = torch.randn(8, 1, 28, 28, device=device)
print(x.shape, x.dtype, x.device)


In [ ]:
# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")


In [ ]:
# Detect NaNs in a tensor
loss = torch.tensor(float("nan"))
print("has NaN:", torch.isnan(loss).any().item())


In [ ]:
# Reproducibility
torch.manual_seed(42)
np.random.seed(42)


### The five mistakes you'll actually make

| # | Mistake | Symptom |
|---|---------|---------|
| 1 | Shape mismatch | `RuntimeError: ... expected size ...` (forgot to flatten before `nn.Linear`) |
| 2 | Forgot `.to(device)` | `RuntimeError: Expected all tensors to be on the same device` |
| 3 | Forgot `optimizer.zero_grad()` | Loss decreases erratically |
| 4 | Wrong dtype on labels | `CrossEntropyLoss` wants `long`; `BCE` wants `float` |
| 5 | Forgot `model.eval()` at evaluation | Test accuracy looks worse than it should |

---

## Wrap-up

You've now run, end-to-end:

- tensor creation and shape manipulation
- autograd
- a custom `nn.Module` (the `MLP`)
- loading MNIST via a `Dataset` + `DataLoader`
- a full train / evaluate / save loop
- a sanity check (overfit one batch) and prediction visualization

This is the entire scaffolding any classification or segmentation pipeline
needs. From here:

1. Replace the `MLP` with whatever architecture your assignment requires.
2. Replace MNIST with the dataset specified by the assignment.
3. Tune hyperparameters (learning rate, epochs, batch size).
4. Iterate on the loss / metrics for your specific task.

**Good luck with the programming assignment:)**
